In [1]:
!git config --global user.name "Brendan Hills"
!git config --global user.email brendanhills@google.com

In [2]:
!python3 -m venv venv
!source venv/bin/activate
%pwd


from platform import python_version

print(python_version())

3.11.2


In [3]:
!poetry install

Installing dependencies from lock file

No dependencies to install or update


In [4]:
!gcloud auth application-default login

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=KldBh0Ic5QqYLy833ujakYu7HXfVIA&access_type=offline&code_challenge=NeOijhGuCme0tWQoFbNci35fGMSujWQYRQH7Wy-eWXA&code_challenge_method=S256


Credentials saved to file: [/home/brendanhills/.config/gcloud/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).

Quota project "uk-bh-experiments-argolis" was added to ADC which can be used by Google client libraries for billing and quota. Note that some services may still bill the project owning the resource.


In [5]:

PROJECT_ID = "uk-bh-experiments-argolis"  # @param {type:"string"}
REGION = "US"  # @param {type: "string"}
DATASET_ID = "schema_mapping"  # @param {type:"string"}

In [6]:
import pandas as pd
from google.cloud import bigquery
from vertexai.language_models import TextEmbeddingModel
import numpy as np
from IPython.display import display, HTML
import matplotlib

In [7]:
def get_table_sample(table_name):
  client = bigquery.Client()
  table_query = f"""
  SELECT * FROM {PROJECT_ID}.{DATASET_ID}.{table_name} TABLESAMPLE SYSTEM (10 PERCENT)
  """
  table_sample = client.query(table_query)
  table_sample_df = table_sample.to_dataframe()
  return table_sample_df




In [8]:
TABLENAME1="insurance"

table1_df = get_table_sample(TABLENAME1)

table1_df.head()



/home/brendanhills/dev/uk-bh-experiments/.venv/lib/python3.11/site-packages/google/cloud/bigquery/table.py:1933: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,age,sex,bmi,children,smoker,region,charges
0,18,female,26.315,0,False,northeast,2198.18985
1,18,female,38.665,2,False,northeast,3393.35635
2,18,female,35.625,0,False,northeast,2211.13075
3,18,female,30.115,0,False,northeast,21344.84670
4,18,male,23.750,0,False,northeast,1705.62450


In [9]:
TABLENAME2="df1_loan"

table2_df = get_table_sample(TABLENAME2)

table2_df.head()


/home/brendanhills/dev/uk-bh-experiments/.venv/lib/python3.11/site-packages/google/cloud/bigquery/table.py:1933: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,int64_field_0,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status,Total_Income
0,63,LP001213,Male,True,1,Graduate,False,4945,0.0,NaN,360.0,0.0,Rural,False,4945.0
1,127,LP001449,Male,False,0,Graduate,False,3865,1640.0,NaN,360.0,1.0,Rural,True,5505.0
2,284,LP001922,Male,True,0,Graduate,False,20667,0.0,NaN,360.0,1.0,Rural,False,20667.0
3,322,LP002054,Male,True,2,Not Graduate,False,3601,1590.0,NaN,360.0,1.0,Rural,True,5191.0
4,231,LP001768,Male,True,0,Graduate,<NA>,3716,0.0,42.0,180.0,1.0,Rural,True,3716.0


In [10]:
def get_embedding_for_col(column):
  embeddings = []
  #print(f'{column=}')
  for row in column:
    embeddings.append(row)
  return embeddings

def get_embeddings_for_table(table):
  embeddings = []
  for col in table.columns:
    embeddings.append(get_embedding_for_col(table[col]))
  return embeddings


def text_embedding(text):
    """Text embedding with a Large Language Model."""
    model = TextEmbeddingModel.from_pretrained("text-embedding-005")
    embeddings = model.get_embeddings(text)
    embedding_vector = []
    for embedding in embeddings:
        embedding_vector.append(embedding.values)
    return embedding_vector

In [11]:
def get_embedding_for_col2(column):
  print(f'{column=}')
  #convert column to list of strings
  col_strings = [str(cell) for cell in column]
  #convert list of strings to string
  col_string = ' '.join(col_strings)
  print(f'{col_string=}')
  embeddings = text_embedding([col_string])
  return embeddings

def get_embeddings_for_table2(table):
  embeddings = []
  for col in table.columns:
    embeddings.append(get_embedding_for_col2(table[col]))
  return embeddings




In [12]:
#embeddings_df1 = pd.DataFrame(get_embeddings_for_table2(table1_df))

#embeddings_df1.head()

In [13]:
## return a dataframe of embeddings - one column for each column of the table
def get_embeddings_for_table_columns(table):
    columns_strings = []
    #convert each column of the table into a list of strings
    for col in table.columns:
        #print(f'{col=}')
        #sort by that column so that the ordering of rows doesn't influence the embedding
        sorted_df = table.sort_values(col)
        #convert column to list of strings
        col_strings = [str(cell) for cell in sorted_df[col]]
        #convert list of strings to string
        col_string =  col + ' ' + ' '.join(col_strings)
        columns_strings.append(col_string)


    column_embeddings = text_embedding(columns_strings)
    embeddings_df = pd.DataFrame( index=table.columns, data=column_embeddings).transpose()
    return embeddings_df
    

In [14]:
embeddings_df1 = get_embeddings_for_table_columns(table1_df)

print ("Embeddings for Table 1:")
display(embeddings_df1)

Embeddings for Table 1:


,age,sex,bmi,children,smoker,region,charges
0,-0.016925,0.019224,-0.007474,-0.010769,-0.001304,0.015867,0.008949
1,-0.005187,-0.020801,0.039227,-0.004251,0.043422,0.024605,0.020619
2,0.006084,0.029349,-0.006982,0.031370,0.016444,0.002403,-0.028587
3,0.014872,0.009451,0.029661,-0.028454,0.003646,0.040512,-0.020768
4,-0.025869,-0.020706,-0.015437,0.006666,-0.019183,-0.012310,0.023298
...,...,...,...,...,...,...,...
763,0.005064,0.011045,-0.004147,-0.019169,-0.007048,0.029429,0.016373
764,0.001244,-0.004413,0.001924,0.001663,0.028514,-0.006545,-0.005945
765,0.010940,-0.031198,-0.005141,0.025481,0.028993,0.015277,0.026811
766,0.036957,0.030836,0.045177,0.038916,0.039948,0.032482,0.033006


In [15]:
embeddings_df2 = get_embeddings_for_table_columns(table2_df)

print ("Embeddings for Table 2:")
display(embeddings_df2)

Embeddings for Table 2:


,int64_field_0,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status,Total_Income
0,0.002148,0.017085,-0.015152,-0.026141,-0.016471,0.011378,-0.010217,-0.019828,-0.002478,0.029821,0.018797,0.006480,0.002372,0.015490,0.016880
1,0.022974,-0.000848,-0.015549,0.015091,-0.014693,-0.013826,0.013654,-0.013101,-0.012845,0.012711,0.025668,0.008648,0.009803,0.026840,-0.023783
2,-0.032134,-0.017531,-0.001130,-0.014458,-0.005578,0.003960,-0.002322,-0.041799,-0.029860,-0.030332,-0.036077,-0.004306,-0.008741,-0.019968,-0.024270
3,-0.007162,-0.012630,0.029615,0.042639,0.005863,0.026027,0.046000,0.002428,-0.002777,-0.006141,-0.019385,-0.020720,-0.009083,0.013185,0.014587
4,0.007394,0.012275,-0.018027,-0.023089,-0.005626,-0.013151,-0.028155,-0.003296,0.015015,0.003631,0.015477,0.020047,0.000223,-0.007972,-0.008113
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
763,-0.006781,-0.011701,-0.015913,-0.015016,-0.014612,-0.004685,-0.017589,-0.003776,0.003727,0.025579,0.018369,-0.029515,-0.001845,-0.006637,0.020832
764,-0.004860,0.017210,0.003208,0.020504,-0.011400,0.019882,0.022004,0.029243,0.024721,0.025253,0.008105,-0.010289,-0.004557,0.043890,0.013136
765,0.048740,0.042395,-0.019088,0.029485,0.039086,0.028127,0.045746,0.031399,0.044540,0.019045,0.031218,0.041592,0.015378,0.044813,0.028607
766,0.032418,0.010614,0.006607,0.002608,0.003000,0.014094,0.012882,0.023684,0.033933,0.028983,0.017639,0.001439,-0.012048,-0.027036,0.021209


In [16]:
def vector_similarity(vec1, vec2):
    return np.dot(np.squeeze(np.array(vec1)),np.squeeze(np.array(vec2)))

In [17]:

#print (f'{distances_df=}')
rows_list = []
for row in embeddings_df1:
    row_distances = []
    for col in embeddings_df2:
        distance = vector_similarity(embeddings_df1[row], embeddings_df2[col])
        #print(f'{distance=}')
        row_distances.append(distance)    
    rows_list.append(row_distances)
        

distances_df = pd.DataFrame(index=embeddings_df1.columns,columns=embeddings_df2.columns, data=rows_list)
print ("Embeddings for Distances:")
distances_df.style \
    .background_gradient(cmap='Blues', axis=None) \
    .format(precision=2)
      


Embeddings for Distances:


,int64_field_0,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status,Total_Income
age,0.78,0.77,0.83,0.79,0.77,0.78,0.79,0.83,0.78,0.79,0.78,0.70,0.71,0.70,0.82
sex,0.67,0.67,0.89,0.79,0.67,0.75,0.77,0.67,0.67,0.66,0.68,0.64,0.70,0.70,0.71
bmi,0.71,0.69,0.74,0.73,0.67,0.67,0.73,0.81,0.75,0.78,0.76,0.61,0.66,0.65,0.82
children,0.77,0.79,0.77,0.76,0.86,0.70,0.75,0.74,0.79,0.73,0.72,0.82,0.68,0.68,0.77
smoker,0.67,0.68,0.77,0.85,0.70,0.69,0.86,0.68,0.68,0.70,0.71,0.64,0.69,0.81,0.72
region,0.61,0.64,0.74,0.67,0.62,0.73,0.71,0.65,0.62,0.64,0.67,0.58,0.80,0.65,0.68
charges,0.72,0.75,0.66,0.66,0.68,0.61,0.67,0.80,0.76,0.79,0.76,0.65,0.62,0.64,0.83


Columns that should be similar:

 insurance.sex, df1_loan.gender
 insurance.children, df1_loan.Dependants